# 06. Итог по флагам 15 / 180, правило end v2 (окно с 2013-01-01)

> **Пересчёт.** Основа — `petri_states_eval_2013.ipynb / petri_states_eval_all.ipynb, шаг 3` из папки `петри`; код расчёта тот же, пути и флаги — из `common/settings.py`. Расчёт — в `03` и `04`; здесь только чтение результатов. Сравнения с вариантом 1 (15 / 365) выполняются, только если он посчитан. Текст с числами и выводами написан по **прежнему** расчёту: после пересчёта сверить (`common/compare_with_old.py`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "common"))
import settings as cfg

PART = "2013"                         # "2013" — с окном, "all" — без окна

import json
import pandas as pd
from IPython.display import display
import petri_states as ps
import align_parallel as ap

CONFIG = "long180_v2"                    # при RUN_V1 = True можно поставить "long180_v1"
LONG_CUSTODY_DAYS = cfg.CONFIGS[CONFIG]["long"]
OUT = cfg.out_dir(PART, CONFIG)
OUT_365 = cfg.out_dir(PART, "long365_v1")
RESULTS = OUT / "split_eval.jsonl"


def done_models():
    return {r["model"]: r for r in ap.jl_read(RESULTS)}

## Итог

In [2]:
res = pd.DataFrame(done_models().values())
q = ps.summarize_split(res)
cols = ["model", "fitness_tbr_train", "fitness_tbr_test", "pct_fit_traces_tbr_test", "fitness_align_test",
        "pct_fit_traces_align_test", "align_timeout", "align_traces", "precision_etc_test", "precision_norm_test",
        "f1_tbr", "f1_align", "places", "transitions", "silent", "total_sec", "finished"]
q[[c for c in cols if c in q]].round(3)

,model,fitness_tbr_train,fitness_tbr_test,pct_fit_traces_tbr_test,fitness_align_test,pct_fit_traces_align_test,align_timeout,align_traces,precision_etc_test,precision_norm_test,f1_tbr,f1_align,places,transitions,silent,finished
0,flower,1.000,1.000,100.000,NaN,NaN,NaN,NaN,0.353,0.000,0.521,NaN,3,12,2,2026-11-05 20:36
1,DFG_p0.05,0.967,0.968,72.417,0.965,72.284,0.0,2255.0,0.756,0.624,0.849,0.848,12,27,1,2026-11-05 20:49
2,DFG_p0.0,1.000,1.000,100.000,1.000,100.000,0.0,2255.0,0.438,0.132,0.609,0.609,12,73,1,2026-11-05 20:49
3,HM_dep0.95,0.982,0.981,60.931,0.944,60.931,0.0,2255.0,0.601,0.384,0.745,0.734,15,34,24,2026-11-05 20:49
4,SplitMiner,0.776,0.779,6.563,0.639,6.563,0.0,2255.0,0.772,0.648,0.775,0.699,19,34,24,2026-11-05 20:49
5,IM_noise0.2,1.000,1.000,99.867,1.000,99.867,0.0,2255.0,0.368,0.023,0.538,0.538,35,44,34,2026-11-05 20:49


## После досчёта тайм-аутов (`04_alignments.ipynb`, шаг 2)

In [3]:
f = OUT / "split_eval_timeouts_fixed.csv"
display(pd.read_csv(f, encoding="utf-8-sig").round(4)) if f.exists() else print("нет", f)

,experiment,model,align_limit_s,traces,timeout_before,timeout_after,fitness_align_before,fitness_align_after,fitness_align_after_upper,pct_fit_traces_before,pct_fit_traces_after,precision_etc_test,precision_norm_test,f1_align_before,f1_align_after,fixed_how
0,2013,DFG_p0.05,120,2255,0,0,0.9652,0.9652,0.9652,72.2838,72.2838,0.7565,0.6238,0.8482,0.8482,NaN
1,2013,DFG_p0.0,120,2255,0,0,1.0000,1.0000,1.0000,100.0000,100.0000,0.4380,0.1319,0.6092,0.6092,NaN
2,2013,HM_dep0.95,120,2255,0,0,0.9437,0.9437,0.9437,60.9313,60.9313,0.6010,0.3836,0.7343,0.7343,NaN
3,2013,SplitMiner,120,2255,0,0,0.6386,0.6386,0.6386,6.5632,6.5632,0.7720,0.6478,0.6990,0.6990,NaN
4,2013,IM_noise0.2,120,2255,0,0,0.9998,0.9998,0.9998,99.8670,99.8670,0.3678,0.0234,0.5378,0.5378,NaN


## Сравнение с расчётом на всём журнале

In [4]:
# сравнение с расчётом без разбиения (01_states_all_miners.ipynb, весь журнал)
f = OUT_365 / "model_quality.csv"     # весь журнал, флаги 15 / 365 (01_states_all_miners.ipynb)
if f.exists():
    full = pd.read_csv(f).set_index("model")[["fitness_tbr", "precision_etc", "precision_norm", "f1"]]
    full.columns = [c + " | весь журнал" for c in full.columns]
    split = q.set_index("model")[["fitness_tbr_test", "fitness_align_test", "precision_etc_test", "precision_norm_test", "f1_tbr", "f1_align"]]
    split.columns = [c + " | test" for c in split.columns]
    display(split.join(full, how="left").round(3))
else:
    print("нет", f, "— сначала выполнить 01_states_all_miners.ipynb")

,fitness_tbr_test | test,fitness_align_test | test,precision_etc_test | test,precision_norm_test | test,f1_tbr | test,f1_align | test,fitness_tbr | весь журнал,precision_etc | весь журнал,precision_norm | весь журнал,f1 | весь журнал
model,,,,,,,,,,
flower,1.000,NaN,0.353,0.000,0.521,NaN,1.000,0.421,0.000,0.592
DFG_p0.05,0.968,0.965,0.756,0.624,0.849,0.848,0.967,0.815,0.681,0.885
DFG_p0.0,1.000,1.000,0.438,0.132,0.609,0.609,1.000,0.527,0.183,0.690
HM_dep0.95,0.981,0.944,0.601,0.384,0.745,0.734,0.982,0.675,0.439,0.800
SplitMiner,0.779,0.639,0.772,0.648,0.775,0.699,0.772,0.813,0.678,0.792
IM_noise0.2,1.000,1.000,0.368,0.023,0.538,0.538,1.000,0.442,0.036,0.613


### Сравнение с прежними флагами (SHORT = 15, LONG = 365)

In [5]:
f = OUT_365 / "split_eval.jsonl"
if f.exists():
    old = ps.summarize_split(pd.DataFrame([json.loads(l) for l in f.read_text(encoding="utf-8").splitlines() if l.strip()]))
    keep = ["fitness_tbr_test", "fitness_align_test", "align_timeout", "precision_etc_test", "precision_norm_test", "f1_tbr", "f1_align"]
    cmp = (q.set_index("model")[[c for c in keep if c in q]].add_suffix(f" | LONG={LONG_CUSTODY_DAYS}")
           .join(old.set_index("model")[[c for c in keep if c in old]].add_suffix(" | LONG=365")))
    cmp = cmp[sorted(cmp.columns, key=lambda c: (keep.index(c.split(" | ")[0]), c))]
    display(cmp.round(3))
else:
    print("нет", f)

,fitness_tbr_test | LONG=180,fitness_tbr_test | LONG=365,fitness_align_test | LONG=180,fitness_align_test | LONG=365,align_timeout | LONG=180,align_timeout | LONG=365,precision_etc_test | LONG=180,precision_etc_test | LONG=365,precision_norm_test | LONG=180,precision_norm_test | LONG=365,f1_tbr | LONG=180,f1_tbr | LONG=365,f1_align | LONG=180,f1_align | LONG=365
model,,,,,,,,,,,,,,
flower,1.000,1.000,NaN,NaN,NaN,NaN,0.353,0.358,0.000,0.000,0.521,0.527,NaN,NaN
DFG_p0.05,0.968,0.968,0.965,0.963,0.0,0.0,0.756,0.759,0.624,0.625,0.849,0.851,0.848,0.849
DFG_p0.0,1.000,NaN,1.000,NaN,0.0,NaN,0.438,NaN,0.132,NaN,0.609,NaN,0.609,NaN
HM_dep0.95,0.981,0.982,0.944,0.944,0.0,0.0,0.601,0.602,0.384,0.381,0.745,0.747,0.734,0.736
SplitMiner,0.779,0.775,0.639,0.633,0.0,0.0,0.772,0.773,0.648,0.647,0.775,0.774,0.699,0.696
IM_noise0.2,1.000,1.000,1.000,1.000,0.0,0.0,0.368,0.374,0.023,0.024,0.538,0.544,0.538,0.544


## ✅ Напоминание: проверить результаты

1. **Все ли модели посчитаны** — в итоговой таблице 5 строк (4 модели + `flower`), у всех заполнено `finished`.
2. **Тайм-ауты alignments**: доля `align_timeout / align_traces`. Если больше ~10% — `fitness_align_test` ненадёжна (в неё входят только «быстрые» трассы); указать это в тексте или пересчитать с большим лимитом / на `N_CORES > 1`.
3. **Переобучение**: `fitness_tbr_train` и `fitness_tbr_test` должны быть близки (разница > 0.02 — сеть подогнана под train).
4. **Сравнение с расчётом на всём журнале** (последняя таблица): precision на test обычно ниже — это нормально; если порядок моделей по F1 поменялся, отметить.
5. **TBR и alignments**: если `fitness_align_test` сильно ниже `fitness_tbr_test` (как у Split Miner), TBR завышает fitness из-за тихих переходов — в выводах опираться на alignments.
6. Записать итоговые числа и выбор лучшей модели в отчёт / `Требования_к_архитектуре.md`.